# Week 4 — Composite comparison index

Combines the three dimensions that are actually comparable on a common 0–100 scale:

| Dimension | Source | Weight |
|---|---|---|
| Cost efficiency score | `cost_efficiency_scores.csv` | 1/3 |
| Proxy-NPS (rescaled to 0–100) | `proxy_nps.csv` | 1/3 |
| Product-depth score | `product_depth_scores.csv` | 1/3 |

**CAC is deliberately excluded** — HDFC has no disclosed CAC, and the two ad-spend-intensity
proxies computed in `02_cac.ipynb` measure different things (whole customer base vs. new
acquisitions), so folding them in would manufacture false precision. See that notebook and
`roadmap.md`'s "Known risks" section for the caveat.

Equal weighting is a starting assumption, not a fitted result — flagged here so it's easy
to revisit once the Week 5 narrative determines which pillar matters most to the thesis.

**Output:** `data/processed/composite_index.csv`

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

cost_eff = pd.read_csv(PROCESSED / "cost_efficiency_scores.csv")[["entity", "cost_efficiency_score"]]
nps = pd.read_csv(PROCESSED / "proxy_nps.csv")[["entity", "proxy_nps"]]
depth = pd.read_csv(PROCESSED / "product_depth_scores.csv")[["entity", "product_depth_score"]]

# cost_efficiency_scores.csv labels Jupiter's entity as "Jupiter"; reviews/feature files use
# the same short name, so no re-mapping needed here (see 01/02/03/04 notebooks for the
# full legal-entity names used in the raw financial data).
cost_eff

,entity,cost_efficiency_score
0,HDFC Bank,59.8
1,Jupiter,0.0


In [2]:
merged = cost_eff.merge(nps, on="entity").merge(depth, on="entity")

# Rescale proxy-NPS (-100..100) onto the same 0..100 footing as the other two dimensions.
merged["proxy_nps_rescaled"] = (merged["proxy_nps"] + 100) / 2

merged["composite_index"] = (
    merged["cost_efficiency_score"] + merged["proxy_nps_rescaled"] + merged["product_depth_score"]
) / 3
merged = merged.round(1)
merged

,entity,cost_efficiency_score,proxy_nps,product_depth_score,proxy_nps_rescaled,composite_index
0,HDFC Bank,59.8,-6.1,95.0,47.0,67.2
1,Jupiter,0.0,-52.9,75.0,23.6,32.8


In [3]:
merged.to_csv(PROCESSED / "composite_index.csv", index=False)
print("-> data/processed/composite_index.csv")
merged[["entity", "cost_efficiency_score", "proxy_nps_rescaled", "product_depth_score", "composite_index"]]

-> data/processed/composite_index.csv


,entity,cost_efficiency_score,proxy_nps_rescaled,product_depth_score,composite_index
0,HDFC Bank,59.8,47.0,95.0,67.2
1,Jupiter,0.0,23.6,75.0,32.8
